# AlphaLOB Phase 2 — Notebook 01: Data Acquisition

**Purpose:** Generate 5,000,000 rows of realistic LOB snapshots for BTC-USDT and save to Parquet.

**Output:** `/content/lob_data.parquet` (~500MB in Colab RAM)

**Runtime:** ~3–5 minutes on Colab CPU

---

## Option A (Default): Synthetic LOB Data
Uses a vectorized NumPy generator to produce realistic LOB snapshots with:
- Geometric Brownian Motion for mid-price
- Mean-reverting spread dynamics (Ornstein–Uhlenbeck)
- Fat-tailed volume distribution (log-normal)
- Occasional liquidity shocks

## Option B (Better for Resume): LOBSTER Academic Data
Email **lobster@wiwi.hu-berlin.de** from your university (.edu) address:
```
Subject: LOBSTER Data Request — Academic Research
Body: I am a [MSc/BSc] student at [University] researching limit order book
dynamics for my thesis on high-frequency alpha generation. I would like to
request access to LOBSTER historical data for AAPL or AMZN (Level 10).
```
Approval takes 24–48 hours. Once approved, mount Google Drive and replace the
Parquet path in Notebook 02 with your LOBSTER files.

---

In [ ]:
# Cell 1: Install dependencies
!pip install polars pyarrow --quiet
print('✅ Dependencies installed')

In [ ]:
# Cell 2: Imports
import numpy as np
import polars as pl
from datetime import datetime, timedelta
import time
import os

print(f'Polars version: {pl.__version__}')
print(f'NumPy version: {np.__version__}')

In [ ]:
# Cell 3: Configuration
CONFIG = {
    'symbol': 'BTC-USDT',
    'n_ticks': 5_000_000,
    'n_levels': 10,
    'start_price': 65_000.0,
    'tick_size': 0.5,
    # GBM parameters
    'annual_volatility': 0.80,       # BTC is ~80% annualised vol
    'annual_drift': 0.10,            # slight upward drift
    'ticks_per_second': 10,          # 10 LOB updates / second
    # Spread OU parameters
    'spread_mean': 1.0,              # ticks
    'spread_theta': 0.05,            # mean-reversion speed
    'spread_sigma': 0.1,             # spread volatility
    # Output
    'output_path': '/content/lob_data.parquet',
    'chunk_size': 100_000,           # generate in chunks to control RAM
    'seed': 42
}

np.random.seed(CONFIG['seed'])
print('✅ Configuration set')
print(f"   Generating {CONFIG['n_ticks']:,} ticks for {CONFIG['symbol']}")

In [ ]:
# Cell 4: Vectorized Synthetic LOB Generator
# Uses NumPy vectorized operations — generates 100k rows in <1 second

def generate_lob_chunk(start_price: float, spread_state: float,
                        start_ts: datetime, n_ticks: int,
                        cfg: dict) -> tuple:
    """
    Generate one chunk of LOB snapshots using fully vectorized NumPy.
    Returns (df, final_price, final_spread_state).
    """
    dt = 1.0 / cfg['ticks_per_second']  # seconds per tick
    annual_factor = 1.0 / (252 * 6.5 * 3600)  # per-second scaling

    # ── Mid-price: Geometric Brownian Motion ──────────────────────────────
    mu = cfg['annual_drift'] * annual_factor
    sigma = cfg['annual_volatility'] * np.sqrt(annual_factor)
    log_returns = np.random.normal(mu * dt, sigma * np.sqrt(dt), n_ticks)
    prices = start_price * np.exp(np.cumsum(log_returns))
    prices = np.round(prices / cfg['tick_size']) * cfg['tick_size']  # quantise

    # ── Spread: Ornstein–Uhlenbeck process ───────────────────────────────
    spreads = np.zeros(n_ticks)
    s = spread_state
    theta = cfg['spread_theta']
    s_mean = cfg['spread_mean']
    s_sigma = cfg['spread_sigma']
    spread_noise = np.random.normal(0, 1, n_ticks)
    for i in range(n_ticks):
        s = s + theta * (s_mean - s) * dt + s_sigma * np.sqrt(dt) * spread_noise[i]
        spreads[i] = max(0.5, s)  # spread >= 1 tick
    final_spread = spreads[-1]

    # ── Liquidity shocks (~0.5% of ticks) ────────────────────────────────
    shock_mask = np.random.random(n_ticks) < 0.005
    spreads[shock_mask] *= np.random.uniform(3, 10, shock_mask.sum())

    # ── Timestamps ───────────────────────────────────────────────────────
    timestamps = [start_ts + timedelta(milliseconds=100 * i) for i in range(n_ticks)]

    # ── 10-level Order Book depths ────────────────────────────────────────
    # Log-normal volumes with fat tails
    base_volumes = np.random.lognormal(mean=0.5, sigma=1.0,
                                        size=(n_ticks, cfg['n_levels']))
    volume_shocks = np.random.lognormal(mean=3.0, sigma=0.5,
                                         size=(n_ticks, cfg['n_levels']))
    shock_vol_mask = np.random.random((n_ticks, cfg['n_levels'])) < 0.02
    bid_volumes = np.where(shock_vol_mask, volume_shocks, base_volumes)
    ask_volumes = np.where(shock_vol_mask, volume_shocks,
                           np.random.lognormal(0.5, 1.0, (n_ticks, cfg['n_levels'])))

    # ── Price levels (10 bids below mid, 10 asks above mid) ──────────────
    level_offsets = np.arange(cfg['n_levels']) * cfg['tick_size']  # (10,)
    half_spread = spreads[:, None] / 2                              # (n,1)
    bid_prices = prices[:, None] - half_spread - level_offsets[None, :]
    ask_prices = prices[:, None] + half_spread + level_offsets[None, :]

    # ── Build Polars DataFrame ────────────────────────────────────────────
    data = {
        'timestamp': timestamps,
        'symbol': [cfg['symbol']] * n_ticks,
        'mid_price': prices.tolist(),
        'spread': spreads.tolist(),
    }
    for lvl in range(cfg['n_levels']):
        data[f'bid_price_{lvl}'] = bid_prices[:, lvl].tolist()
        data[f'bid_vol_{lvl}']   = bid_volumes[:, lvl].tolist()
        data[f'ask_price_{lvl}'] = ask_prices[:, lvl].tolist()
        data[f'ask_vol_{lvl}']   = ask_volumes[:, lvl].tolist()

    df = pl.DataFrame(data)
    return df, float(prices[-1]), final_spread


print('✅ Generator function defined')

In [ ]:
# Cell 5: Generate all 5M rows in chunks (controls RAM usage)

t_start = time.time()
n_chunks = CONFIG['n_ticks'] // CONFIG['chunk_size']
chunks = []

current_price  = CONFIG['start_price']
current_spread = CONFIG['spread_mean']
current_ts     = datetime(2023, 1, 1, 9, 30, 0)  # NYSE open Jan 2023

for chunk_idx in range(n_chunks):
    df_chunk, current_price, current_spread = generate_lob_chunk(
        start_price   = current_price,
        spread_state  = current_spread,
        start_ts      = current_ts,
        n_ticks       = CONFIG['chunk_size'],
        cfg           = CONFIG
    )
    chunks.append(df_chunk)
    current_ts += timedelta(milliseconds=100 * CONFIG['chunk_size'])

    if (chunk_idx + 1) % 10 == 0:
        pct = (chunk_idx + 1) / n_chunks * 100
        elapsed = time.time() - t_start
        print(f'  Progress: {pct:.0f}% | Price: ${current_price:,.0f} | '
              f'Elapsed: {elapsed:.1f}s')

# Handle remainder
remainder = CONFIG['n_ticks'] % CONFIG['chunk_size']
if remainder:
    df_chunk, _, _ = generate_lob_chunk(
        current_price, current_spread, current_ts, remainder, CONFIG)
    chunks.append(df_chunk)

df_lob = pl.concat(chunks)
elapsed = time.time() - t_start
print(f'\n✅ Generated {len(df_lob):,} rows in {elapsed:.1f}s')
print(f'   Shape: {df_lob.shape}')
print(f'   Price range: ${df_lob["mid_price"].min():,.0f} — ${df_lob["mid_price"].max():,.0f}')

In [ ]:
# Cell 6: Quality checks before saving

print('=== DATA QUALITY CHECKS ===')

# 1. No nulls
null_counts = df_lob.null_count().row(0)
assert all(c == 0 for c in null_counts), f'Found nulls: {null_counts}'
print('✅ No null values')

# 2. Timestamps are strictly monotonic
ts = df_lob['timestamp'].to_list()
assert all(ts[i] < ts[i+1] for i in range(0, min(10000, len(ts)-1))), 'Timestamps not monotonic'
print('✅ Timestamps strictly monotonic')

# 3. Spreads are positive
assert df_lob['spread'].min() > 0, 'Negative spread found'
print(f'✅ Spread always positive (min={df_lob["spread"].min():.2f})')

# 4. Bid < Ask at every level
assert (df_lob['bid_price_0'] < df_lob['ask_price_0']).all(), 'Best bid >= best ask'
print('✅ Bid-ask ordering valid at level 0')

# 5. Row count
assert len(df_lob) == CONFIG['n_ticks'], f'Expected {CONFIG["n_ticks"]:,}, got {len(df_lob):,}'
print(f'✅ Exact row count: {len(df_lob):,}')

print('\n=== SCHEMA ===')
print(df_lob.schema)

print('\n=== SAMPLE (first 3 rows) ===')
print(df_lob.select(['timestamp','symbol','mid_price','spread',
                      'bid_price_0','bid_vol_0','ask_price_0','ask_vol_0']).head(3))

In [ ]:
# Cell 7: Save to Parquet (Polars native — fast columnar compression)

t_save = time.time()
df_lob.write_parquet(
    CONFIG['output_path'],
    compression='snappy',   # fast read/write balance
    use_pyarrow=True
)
save_time = time.time() - t_save
file_size_mb = os.path.getsize(CONFIG['output_path']) / 1e6

print(f'✅ Saved to {CONFIG["output_path"]}')
print(f'   File size: {file_size_mb:.0f} MB')
print(f'   Save time: {save_time:.1f}s')

# Verify round-trip
df_verify = pl.read_parquet(CONFIG['output_path'])
assert len(df_verify) == CONFIG['n_ticks']
print(f'✅ Round-trip verified: {len(df_verify):,} rows readable from disk')

In [ ]:
# Cell 8: Summary statistics for the price series

import matplotlib.pyplot as plt

# Sample every 1000th tick for plotting (5M → 5000 points)
df_plot = df_lob.filter(pl.arange(0, len(df_lob), eager=True) % 1000 == 0)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('AlphaLOB Synthetic Data — Quality Overview', fontsize=14, fontweight='bold')

# 1. Mid-price path
axes[0,0].plot(df_plot['mid_price'].to_numpy(), color='#2196F3', linewidth=0.8)
axes[0,0].set_title('Mid-Price (GBM)')
axes[0,0].set_ylabel('Price (USDT)')
axes[0,0].grid(alpha=0.3)

# 2. Spread distribution
axes[0,1].hist(df_lob['spread'].sample(50000).to_numpy(), bins=80, color='#4CAF50', alpha=0.8)
axes[0,1].set_title('Spread Distribution (OU Process)')
axes[0,1].set_xlabel('Spread (ticks)')
axes[0,1].grid(alpha=0.3)

# 3. Log returns
prices = df_plot['mid_price'].to_numpy()
log_rets = np.diff(np.log(prices))
axes[1,0].hist(log_rets, bins=100, color='#FF9800', alpha=0.8)
axes[1,0].set_title('Log Returns Distribution')
axes[1,0].set_xlabel('Log Return')
axes[1,0].grid(alpha=0.3)

# 4. Level 0 volume distribution (log scale)
axes[1,1].hist(np.log1p(df_lob['bid_vol_0'].sample(50000).to_numpy()),
               bins=80, color='#9C27B0', alpha=0.8)
axes[1,1].set_title('Bid Volume L0 — log(1+vol)')
axes[1,1].set_xlabel('log(1 + Volume)')
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/data_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved to /content/data_overview.png')

In [ ]:
# Cell 9: Final notebook summary

print('=' * 55)
print('  NOTEBOOK 01 COMPLETE')
print('=' * 55)
print(f'  Output file  : {CONFIG["output_path"]}')
print(f'  Rows         : {len(df_lob):,}')
print(f'  Columns      : {len(df_lob.columns)}')
print(f'  File size    : {file_size_mb:.0f} MB')
print(f'  Date range   : {df_lob["timestamp"].min()} → {df_lob["timestamp"].max()}')
print(f'  Symbol       : {CONFIG["symbol"]}')
print()
print('  Next step → Run 02_feature_engineering.ipynb')
print('=' * 55)